In [1]:
import os
import gc
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, log_loss
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.decomposition import PCA

import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from scipy.stats import randint, uniform

# Settings
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

IMG_SIZE = 224
BATCH_SIZE = 32
N = 10   # ways
K = 3   # shots
Q = 4   # queries per class
EPISODES = 1000  # Equivalent to few epochs

In [2]:
# Dataset 
class ProtoDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = np.array(file_paths)
        self.labels = np.array(labels)
        self.transform = transform
        self.classes = np.unique(labels)
        self.class_to_indices = {c: np.where(labels == c)[0] for c in self.classes}

    def sample_episode(self, N, K, Q):
        """Create one training episode."""
        selected_classes = np.random.choice(self.classes, N, replace=False)

        support_images = []
        support_labels = []
        query_images = []
        query_labels = []

        for i, cls in enumerate(selected_classes):
            idxs = np.random.choice(self.class_to_indices[cls], K + Q, replace=False)

            support_idxs = idxs[:K]
            query_idxs = idxs[K:]

            for idx in support_idxs:
                img = Image.open(self.file_paths[idx]).convert("RGB")
                if self.transform: img = self.transform(img)
                support_images.append(img)
                support_labels.append(i)

            for idx in query_idxs:
                img = Image.open(self.file_paths[idx]).convert("RGB")
                if self.transform: img = self.transform(img)
                query_images.append(img)
                query_labels.append(i)

        return (
            torch.stack(support_images),
            torch.tensor(support_labels),
            torch.stack(query_images),
            torch.tensor(query_labels)
        )

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [5]:
# Load Dataset
# Define DATA_PATH and CLASSES
DATA_PATH = "/kaggle/input/datasets/sabrina001/eye-disease-image-dataset/Original Dataset"
CLASSES = sorted(os.listdir(DATA_PATH))

file_paths, labels = [], []
for idx, cls in enumerate(CLASSES):
    cls_path = os.path.join(DATA_PATH, cls)
    for f in os.listdir(cls_path):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            file_paths.append(os.path.join(cls_path, f))
            labels.append(idx)

file_paths = np.array(file_paths)
labels = np.array(labels)
# Train/Val/Test split
X_train, X_temp, y_train, y_temp = train_test_split(file_paths, labels, test_size=0.3, stratify=labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

print(f"Train_size: {len(X_train)}\nVal_size: {len(X_val)}\nTest_size: {len(X_test)}")

from collections import Counter

# Count samples
train_counts = Counter(y_train)
val_counts = Counter(y_val)
test_counts = Counter(y_test)

print(f"{'Class':<50} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 48)

for idx, cls in enumerate(CLASSES):
    print(f"{cls:<50} {train_counts[idx]:>8} {val_counts[idx]:>8} {test_counts[idx]:>8}")

Train_size: 3734
Val_size: 800
Test_size: 801
Class                                                 Train      Val     Test
------------------------------------------------
Central Serous Chorioretinopathy (Color Fundus)          71       15       15
Diabetic Retinopathy                                   1056      226      227
Disc Edema                                               89       19       19
Glaucoma                                                944      202      203
Healthy                                                 717      153      154
Macular Scar                                            311       67       66
Myopia                                                  350       75       75
Pterygium                                                12        3        2
Retinal Detachment                                       87       19       19
Retinitis Pigmentosa                                     97       21       21
